# Extract Geode file times and append them to Jochen metadata workbook

This notebook scans the Geode field-data folders, reads the embedded start time from each `*.dat` file, and appends those times to the relevant acquisition sheets in `jochen_field_notes_metadata_tables.xlsx`.

The Geode laptop clock was **not UTC**, so the extracted time should be treated as **Geode laptop time** until we estimate the offset from the nodal data.

Main outputs:

1. `geode_file_times.csv` — one row per discovered `.dat` file.
2. `jochen_field_notes_metadata_tables_with_geode_times.xlsx` — a copy of Jochen's workbook with added columns in each shot/acquisition sheet.

This notebook does not modify the original workbook unless you explicitly point `OUTPUT_XLSX` at the same file.

In [1]:
from pathlib import Path
from datetime import datetime, date
import re
import shutil
import warnings

import numpy as np
import pandas as pd
import openpyxl
import obspy
from obspy import UTCDateTime

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

## User configuration

Update these paths on the Mac mini if needed. The data root is expected to contain folders such as `LBS_051626...` or `LBSSP_051626...`.

In [2]:
# Geode raw-data root on tachyon.
GEODE_DATA_ROOT = Path("/Volumes/tachyon/LBSSP_DATA/GEODE_DATA")

# Jochen's digitized field-note workbook.
# In the project repo this may live under 04_FinalReport/seismic_analysis.
METADATA_XLSX = Path("/Volumes/tachyon/LBSSP_DATA/metadata/jochen_field_notes_metadata_tables.xlsx")

# If the path above does not exist, this notebook also checks the current folder.
METADATA_XLSX_FALLBACKS = [
    Path.cwd() / "jochen_field_notes_metadata_tables.xlsx",
    Path("jochen_field_notes_metadata_tables.xlsx"),
]

# Output files. These are safe copies/products.
OUTPUT_DIR = Path("./geode_time_metadata_outputs")
OUTPUT_CSV = OUTPUT_DIR / "geode_file_times.csv"
OUTPUT_XLSX = OUTPUT_DIR / "jochen_field_notes_metadata_tables_with_geode_times.xlsx"

# Acquisition sheets in Jochen's workbook that contain a file_no column.
METADATA_SHEETS = [
    "T1_1m_Refraction",
    "T1_2m_Refraction",
    "T1_Streamer_MASW",
    "T1A_Streamer_MASW",
    "T3_1m_Refraction",
    "T4_1m_Refraction",
]

# Folder prefixes to scan.
FOLDER_PREFIXES = ("LBS", "LBSSP")

# Reading controls.
READ_FORMATS_TO_TRY = ("SEG2", None)  # None lets ObsPy autodetect.
MAX_FILES_FOR_TEST = None             # e.g. set to 10 for a quick smoke test.
OVERWRITE_OUTPUTS = True

## Helper functions

In [3]:
def resolve_existing_path(primary: Path, fallbacks=()) -> Path:
    candidates = [Path(primary).expanduser(), *[Path(p).expanduser() for p in fallbacks]]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError("Could not find file. Checked:\n" + "\n".join(f"  - {p}" for p in candidates))


def parse_date_from_geode_folder(folder: Path) -> date | None:
    # Parse the first plausible MMDDYY token from the Geode folder name.
    match = re.search(r"(?<!\d)(\d{6})(?!\d)", folder.name)
    if not match:
        return None
    mdy = match.group(1)
    mm, dd, yy = int(mdy[:2]), int(mdy[2:4]), int(mdy[4:])
    try:
        return date(2000 + yy, mm, dd)
    except ValueError:
        return None


def find_geode_folders(root: Path, prefixes=FOLDER_PREFIXES) -> list[Path]:
    root = Path(root).expanduser()
    if not root.exists():
        raise FileNotFoundError(root)
    return sorted([item for item in root.iterdir() if item.is_dir() and any(item.name.upper().startswith(prefix.upper()) for prefix in prefixes)])


def find_dat_files(root: Path, prefixes=FOLDER_PREFIXES) -> list[Path]:
    datfiles = []
    for folder in find_geode_folders(root, prefixes=prefixes):
        datfiles.extend(sorted(folder.glob("*.dat")))
    return sorted(datfiles)


def read_geode_dat_metadata(datfile: Path) -> dict:
    # Read lightweight metadata from a Geode SEG-2 .dat file. The extracted time is not corrected to UTC.
    datfile = Path(datfile)
    row = {
        "file_no": pd.NA,
        "datfile": datfile.name,
        "datfile_stem": datfile.stem,
        "geode_folder": datfile.parent.name,
        "geode_folder_date": parse_date_from_geode_folder(datfile.parent),
        "geode_file_path": str(datfile),
        "read_ok": False,
        "read_format": None,
        "error": None,
        "n_traces": pd.NA,
        "sampling_rate_hz": np.nan,
        "npts_first_trace": pd.NA,
        "duration_s_first_trace": np.nan,
        "geode_laptop_starttime": pd.NaT,
        "geode_laptop_starttime_iso": pd.NA,
        "geode_laptop_starttime_epoch": np.nan,
        "geode_time_source": pd.NA,
    }

    try:
        row["file_no"] = int(datfile.stem)
    except Exception:
        pass

    last_error = None
    for fmt in READ_FORMATS_TO_TRY:
        try:
            kwargs = {"headonly": True}
            if fmt is not None:
                kwargs["format"] = fmt
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                st = obspy.read(str(datfile), **kwargs)
            if len(st) == 0:
                raise ValueError("ObsPy returned an empty Stream")

            tr0 = st[0]
            start = tr0.stats.starttime
            sr = float(tr0.stats.sampling_rate)
            npts = int(tr0.stats.npts)
            duration = (npts - 1) / sr if sr > 0 and npts > 0 else np.nan

            row.update({
                "read_ok": True,
                "read_format": fmt or "autodetect",
                "n_traces": len(st),
                "sampling_rate_hz": sr,
                "npts_first_trace": npts,
                "duration_s_first_trace": duration,
                "geode_laptop_starttime": start.datetime.replace(tzinfo=None),
                "geode_laptop_starttime_iso": start.isoformat(),
                "geode_laptop_starttime_epoch": float(start.timestamp),
                "geode_time_source": "obspy_trace_stats_starttime",
                "error": None,
            })
            return row
        except Exception as exc:
            last_error = exc

    row["error"] = repr(last_error)
    return row


def extract_all_geode_file_times(geode_root: Path, max_files: int | None = MAX_FILES_FOR_TEST) -> pd.DataFrame:
    datfiles = find_dat_files(geode_root, prefixes=FOLDER_PREFIXES)
    if max_files is not None:
        datfiles = datfiles[:max_files]
    print(f"Found {len(datfiles)} .dat files")

    rows = []
    for i, datfile in enumerate(datfiles, start=1):
        if i == 1 or i % 25 == 0 or i == len(datfiles):
            print(f"[{i}/{len(datfiles)}] {datfile}")
        rows.append(read_geode_dat_metadata(datfile))

    df = pd.DataFrame(rows)
    if len(df):
        df = df.sort_values(["geode_folder_date", "file_no", "geode_file_path"], na_position="last").reset_index(drop=True)
    return df

## Extract Geode file times

This scans all matching folders and reads only headers. If you want a quick test first, set `MAX_FILES_FOR_TEST = 10` in the configuration cell.

In [4]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GEODE_DATA_ROOT = Path(GEODE_DATA_ROOT).expanduser()
METADATA_XLSX = resolve_existing_path(METADATA_XLSX, METADATA_XLSX_FALLBACKS)

print(f"GEODE_DATA_ROOT = {GEODE_DATA_ROOT}")
print(f"METADATA_XLSX   = {METADATA_XLSX}")

df_times = extract_all_geode_file_times(GEODE_DATA_ROOT, max_files=MAX_FILES_FOR_TEST)
display(df_times.head())
print(df_times["read_ok"].value_counts(dropna=False))

if OVERWRITE_OUTPUTS or not OUTPUT_CSV.exists():
    df_times.to_csv(OUTPUT_CSV, index=False)
    print(f"Wrote {OUTPUT_CSV}")
else:
    print(f"CSV exists, not overwriting: {OUTPUT_CSV}")

GEODE_DATA_ROOT = /Volumes/tachyon/LBSSP_DATA/GEODE_DATA
METADATA_XLSX   = /Volumes/tachyon/LBSSP_DATA/metadata/jochen_field_notes_metadata_tables.xlsx
Found 326 .dat files
[1/326] /Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_051826/._3022.dat
[25/326] /Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_051826/3027.dat
[50/326] /Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_051826/3052.dat
[75/326] /Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_051826/3077.dat
[100/326] /Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_051926/4014.dat
[125/326] /Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_051926/4039.dat
[150/326] /Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_051926/4064.dat
[175/326] /Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBS_051626/1019.dat
[200/326] /Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBS_051626/1044.dat
[225/326] /Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBS_051626/1069.dat
[250/326] /Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBS_051726/2011.dat
[275/326] /Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBS_0517

,file_no,datfile,datfile_stem,geode_folder,geode_folder_date,geode_file_path,read_ok,read_format,error,n_traces,sampling_rate_hz,npts_first_trace,duration_s_first_trace,geode_laptop_starttime,geode_laptop_starttime_iso,geode_laptop_starttime_epoch,geode_time_source
0,1001,1001.dat,1001,LBS_051626,2026-05-16,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBS_051...,True,SEG2,None,24,2000.0,4000,1.9995,2026-05-16 13:12:26,2026-05-16T13:12:26,1.778937e+09,obspy_trace_stats_starttime
1,1002,1002.dat,1002,LBS_051626,2026-05-16,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBS_051...,True,SEG2,None,24,2000.0,4000,1.9995,2026-05-16 14:20:53,2026-05-16T14:20:53,1.778941e+09,obspy_trace_stats_starttime
2,1003,1003.dat,1003,LBS_051626,2026-05-16,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBS_051...,True,SEG2,None,24,2000.0,4000,1.9995,2026-05-16 14:24:44,2026-05-16T14:24:44,1.778941e+09,obspy_trace_stats_starttime
3,1004,1004.dat,1004,LBS_051626,2026-05-16,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBS_051...,True,SEG2,None,24,2000.0,4000,1.9995,2026-05-16 14:28:19,2026-05-16T14:28:19,1.778942e+09,obspy_trace_stats_starttime
4,1005,1005.dat,1005,LBS_051626,2026-05-16,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBS_051...,True,SEG2,None,24,2000.0,4000,1.9995,2026-05-16 14:32:02,2026-05-16T14:32:02,1.778942e+09,obspy_trace_stats_starttime


read_ok
True     324
False      2
Name: count, dtype: int64
Wrote geode_time_metadata_outputs/geode_file_times.csv


## Check for duplicate file numbers

File number alone should usually identify a shot stack. If the same `file_no` appears in multiple folders, the notebook keeps all candidates in the CSV, but only appends a single time to the workbook when it can choose unambiguously.

In [5]:
if len(df_times):
    dup = df_times[df_times.duplicated("file_no", keep=False)].sort_values(["file_no", "geode_folder"])
    print(f"Duplicate file_no rows: {len(dup)}")
    display(dup[["file_no", "geode_folder", "geode_laptop_starttime_iso", "read_ok", "geode_file_path", "error"]].head(50))

Duplicate file_no rows: 2


,file_no,geode_folder,geode_laptop_starttime_iso,read_ok,geode_file_path,error
254,<NA>,LBSSP_051826,<NA>,False,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,TypeError('Unknown format for file /Volumes/ta...
255,<NA>,LBSSP_051826,<NA>,False,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,TypeError('Unknown format for file /Volumes/ta...


## Append Geode times to Jochen workbook

This makes a copy of Jochen's workbook and adds columns to the right side of each acquisition sheet. Existing formatting and sheets are preserved as much as possible.

In [9]:
APPENDED_COLUMNS = [
    "geode_laptop_starttime",
    "geode_laptop_starttime_iso",
    "geode_folder_date",
    "geode_folder",
    "geode_file_path",
    "geode_read_ok",
    "geode_read_format",
    "geode_n_traces",
    "geode_sampling_rate_hz",
    "geode_duration_s_first_trace",
    "geode_time_source",
    "geode_match_status",
    "geode_match_note",
]


def build_file_time_lookup(df_times: pd.DataFrame) -> dict[int, dict]:
    lookup = {}
    if df_times.empty or "file_no" not in df_times.columns:
        return lookup

    for file_no, group in df_times.dropna(subset=["file_no"]).groupby("file_no"):
        file_no = int(file_no)
        good = group[group["read_ok"] == True].copy()
        if len(group) == 1:
            rec = group.iloc[0].to_dict()
            rec["geode_match_status"] = "matched"
            rec["geode_match_note"] = "single file_no match"
        elif len(good) == 1:
            rec = good.iloc[0].to_dict()
            rec["geode_match_status"] = "matched_duplicate_file_no_single_successful_read"
            rec["geode_match_note"] = f"{len(group)} files share this file_no; selected the only successful read"
        else:
            rec = group.iloc[0].to_dict()
            rec["geode_match_status"] = "ambiguous_duplicate_file_no"
            rec["geode_match_note"] = f"{len(group)} files share this file_no; inspect geode_file_times.csv"
        lookup[file_no] = rec
    return lookup


def normalize_header(value) -> str:
    return str(value).strip().lower().replace(" ", "_").replace("-", "_") if value is not None else ""


def find_header_row_and_file_no_col(ws, max_header_rows: int = 5):
    for r in range(1, min(max_header_rows, ws.max_row) + 1):
        headers = [normalize_header(ws.cell(r, col).value) for col in range(1, ws.max_column + 1)]
        if "file_no" in headers:
            return r, headers.index("file_no") + 1
    return None, None


from copy import copy

def ensure_appended_columns(ws, header_row: int, columns: list[str]) -> dict[str, int]:
    existing = {
        normalize_header(ws.cell(header_row, col).value): col
        for col in range(1, ws.max_column + 1)
    }

    colmap = {}
    next_col = ws.max_column + 1

    for name in columns:
        key = normalize_header(name)

        if key in existing:
            colmap[name] = existing[key]
            continue

        col = next_col
        src = ws.cell(header_row, max(1, col - 1))
        dst = ws.cell(header_row, col)

        dst.value = name

        if src.has_style:
            dst._style = copy(src._style)
            dst.font = copy(src.font)
            dst.fill = copy(src.fill)
            dst.border = copy(src.border)
            dst.alignment = copy(src.alignment)
            dst.number_format = src.number_format
            dst.protection = copy(src.protection)

        colmap[name] = col
        existing[key] = col
        next_col += 1

    return colmap


def excel_safe(value):
    try:
        if value is None or pd.isna(value):
            return None
    except Exception:
        pass
    if isinstance(value, pd.Timestamp):
        return value.to_pydatetime().replace(tzinfo=None)
    if isinstance(value, UTCDateTime):
        return value.datetime.replace(tzinfo=None)
    if isinstance(value, np.generic):
        return value.item()
    return value


def append_geode_times_to_workbook(metadata_xlsx: Path, df_times: pd.DataFrame, output_xlsx: Path) -> pd.DataFrame:
    output_xlsx = Path(output_xlsx)
    output_xlsx.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(metadata_xlsx, output_xlsx)
    wb = openpyxl.load_workbook(output_xlsx)
    lookup = build_file_time_lookup(df_times)
    summary_rows = []

    for sheet in METADATA_SHEETS:
        if sheet not in wb.sheetnames:
            summary_rows.append({"sheet": sheet, "status": "missing_sheet", "matched": 0, "unmatched": 0, "ambiguous": 0})
            continue
        ws = wb[sheet]
        header_row, file_no_col = find_header_row_and_file_no_col(ws)
        if header_row is None:
            summary_rows.append({"sheet": sheet, "status": "no_file_no_header", "matched": 0, "unmatched": 0, "ambiguous": 0})
            continue

        colmap = ensure_appended_columns(ws, header_row, APPENDED_COLUMNS)
        matched = unmatched = ambiguous = 0

        for r in range(header_row + 1, ws.max_row + 1):
            raw_file_no = ws.cell(r, file_no_col).value
            if raw_file_no is None or str(raw_file_no).strip() == "":
                continue
            try:
                file_no = int(raw_file_no)
            except Exception:
                continue

            rec = lookup.get(file_no)
            if rec is None:
                unmatched += 1
                values = {"geode_match_status": "no_matching_dat_file", "geode_match_note": "No .dat file with this file_no was found under GEODE_DATA_ROOT"}
            else:
                status = rec.get("geode_match_status", "matched")
                if str(status).startswith("ambiguous"):
                    ambiguous += 1
                else:
                    matched += 1
                values = {
                    "geode_laptop_starttime": rec.get("geode_laptop_starttime"),
                    "geode_laptop_starttime_iso": rec.get("geode_laptop_starttime_iso"),
                    "geode_folder_date": rec.get("geode_folder_date"),
                    "geode_folder": rec.get("geode_folder"),
                    "geode_file_path": rec.get("geode_file_path"),
                    "geode_read_ok": rec.get("read_ok"),
                    "geode_read_format": rec.get("read_format"),
                    "geode_n_traces": rec.get("n_traces"),
                    "geode_sampling_rate_hz": rec.get("sampling_rate_hz"),
                    "geode_duration_s_first_trace": rec.get("duration_s_first_trace"),
                    "geode_time_source": rec.get("geode_time_source"),
                    "geode_match_status": status,
                    "geode_match_note": rec.get("geode_match_note"),
                }

            for name, value in values.items():
                cell = ws.cell(r, colmap[name])
                cell.value = excel_safe(value)
                if name == "geode_laptop_starttime":
                    cell.number_format = "yyyy-mm-dd hh:mm:ss.000"
                elif name == "geode_folder_date":
                    cell.number_format = "yyyy-mm-dd"

        for name, col in colmap.items():
            width = 22 if "time" in name or "path" in name or "note" in name else 16
            ws.column_dimensions[openpyxl.utils.get_column_letter(col)].width = min(width, 60)

        summary_rows.append({"sheet": sheet, "status": "updated", "matched": matched, "unmatched": unmatched, "ambiguous": ambiguous})

    summary_sheet = "Geode_Time_Extraction_Summary"
    if summary_sheet in wb.sheetnames:
        del wb[summary_sheet]
    ws = wb.create_sheet(summary_sheet)
    ws.append(["item", "value"])
    ws.append(["source_metadata_xlsx", str(metadata_xlsx)])
    ws.append(["geode_data_root", str(GEODE_DATA_ROOT)])
    ws.append(["csv_output", str(OUTPUT_CSV)])
    ws.append(["created", datetime.now().isoformat(timespec="seconds")])
    ws.append(["note", "Geode laptop times are not corrected to UTC."])
    ws.append([])
    ws.append(["sheet", "status", "matched", "unmatched", "ambiguous"])
    for row in summary_rows:
        ws.append([row["sheet"], row["status"], row["matched"], row["unmatched"], row["ambiguous"]])
    ws.column_dimensions["A"].width = 32
    ws.column_dimensions["B"].width = 120
    wb.save(output_xlsx)
    return pd.DataFrame(summary_rows)

In [10]:
summary_update = append_geode_times_to_workbook(METADATA_XLSX, df_times, OUTPUT_XLSX)
display(summary_update)
print(f"Wrote updated workbook: {OUTPUT_XLSX}")

,sheet,status,matched,unmatched,ambiguous
0,T1_1m_Refraction,updated,42,4,0
1,T1_2m_Refraction,updated,42,0,0
2,T1_Streamer_MASW,updated,0,0,0
3,T1A_Streamer_MASW,updated,87,0,0
4,T3_1m_Refraction,updated,0,0,0
5,T4_1m_Refraction,updated,14,0,0


Wrote updated workbook: geode_time_metadata_outputs/jochen_field_notes_metadata_tables_with_geode_times.xlsx


## Quick review

Use this to inspect unmatched or ambiguous records before trusting the workbook columns.

In [11]:
failed = df_times[df_times["read_ok"] != True]
print(f"Failed file reads: {len(failed)}")
display(failed[["file_no", "geode_folder", "datfile", "error", "geode_file_path"]].head(50))

dup = df_times[df_times.duplicated("file_no", keep=False)].sort_values(["file_no", "geode_folder"])
print(f"Duplicate file_no rows: {len(dup)}")
display(dup[["file_no", "geode_folder", "datfile", "geode_laptop_starttime_iso", "read_ok", "geode_file_path"]].head(50))

Failed file reads: 2


,file_no,geode_folder,datfile,error,geode_file_path
254,<NA>,LBSSP_051826,._3022.dat,TypeError('Unknown format for file /Volumes/ta...,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...
255,<NA>,LBSSP_051826,._3024.dat,TypeError('Unknown format for file /Volumes/ta...,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...


Duplicate file_no rows: 2


,file_no,geode_folder,datfile,geode_laptop_starttime_iso,read_ok,geode_file_path
254,<NA>,LBSSP_051826,._3022.dat,<NA>,False,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...
255,<NA>,LBSSP_051826,._3024.dat,<NA>,False,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...


## Next step: active-source time windows for nodal processing

After this workbook contains Geode laptop times, the next notebook can:

1. compare each Geode stack time with the nearest nodal detection time;
2. estimate the Geode-laptop-to-UTC offset for each day/session;
3. create a compact table of active-source UTC windows; and
4. restrict the chunked nodal shot-gather notebook to only those active-source windows.

A conservative first window might be something like `first_stack_time - 30 s` to `last_stack_time + 30 s` per survey block, once the UTC offset is estimated.

In [12]:
print(OUTPUT_XLSX)

geode_time_metadata_outputs/jochen_field_notes_metadata_tables_with_geode_times.xlsx
